In [15]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx, WatsonxEmbeddings
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
# .env 내용 가져오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")

from dotenv import load_dotenv


load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
    temperature=0,
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)


C:\Users\soldesk\AppData\Local\Temp\ipykernel_22344\1028210949.py:29: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


#### LangGraph

- 워크 플로 프레임워크
- LCEL 선형 (A->B->C) / LangGraph 순환 (A-B-A-C-B) 같은 흐름 지원
- 개념
    - node : 실행 할 함수
    - edge : 노드 간 연결
    - stage: 전체 상태를 담는 딕셔너리
- 조건부 엣지, Agent 루프, 자기 수정, 멀티 에이전트 등 복잡한 패턴 구현

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import Literal, NotRequired, Required, TypedDict


### TypeDict / Pydantic
- typeddict : 딕셔너리 키, 값 타입을 정의 할 수 있게 도와줌, 단순 State 구현 시 사용, 기본값 줄 수 없음
- Pydantic : validation 검사

In [12]:
# 데이터 저장소 생성


class MyState(TypedDict):
    message: str


def say_hello(state):
    return {"message": "Hello, LangGraph!"}


# 그래프 생성
graph = StateGraph(MyState)
graph.add_node("hello", say_hello)

graph.add_edge(START, "hello")
graph.add_edge("hello", END)


# 그래프 실행
app = graph.compile()
result = app.invoke({"message": ""})
print(result)

# 그래프 시각화 (참고)
app.get_graph().print_ascii()

{'message': 'Hello, LangGraph!'}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | hello |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


### State
- 상태를 중심으로 동작
- TypeDict, Pydantic Base Model을 사용하여 정의
- 그래프 실행 중 지속적으로 업데이트 됨
- 노드간의 전환은 조건부 엣지를 통해 제어 가능하며 복잡한 의사결정 프로세스 모델링 가능
- 재귀적 실행 지원

### Node
- 실제 작업을 수행하는 기본단위
- 함수 기반
- 상태 중심 : 현재 상태를 입력으로 받아 처리
- 독립적 실행 : 각 노드는 독립적으로 실행
- 조합 가능 : 여러 노드를 연결하여 복잡한 워크 플로 가능

### Edge

- 노드 간의 연결과 실행 흐름을 정의

In [13]:
# 카운터 공유
class CounterState(TypedDict):
    count: int


# 증가 함수
def increment(state):
    print(f"현재 카운트 : {state['count']}")
    new_count = state["count"] + 1
    print(f"새로운 카운트 : {new_count}")

    # state 값 변경 (return)
    return {"count": new_count}


# 그래프 생성
graph = StateGraph(CounterState)
graph.add_node("increment", increment)

# 그래프 연결 (그래프가 실행될 것인가)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# 그래프를 실행 가능한 형태로 변경
app = graph.compile()
result = app.invoke({"count": 0})
print(f"최종 결과 {result}")

app.get_graph().print_ascii()

현재 카운트 : 0
새로운 카운트 : 1
최종 결과 {'count': 1}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| increment |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [14]:
class CounterState(TypedDict):
    count: int


graph = StateGraph(CounterState)

# 2개의 노드


def first_increment(state):
    # +1
    return {"count": state["count"] + 1}


def second_increment(state):
    # +10
    return {"count": state["count"] + 10}


# START => first => second => END
graph.add_node("increment1", first_increment)
graph.add_node("increment2", second_increment)

graph.add_edge(START, "increment1")
graph.add_edge("increment1", "increment2")
graph.add_edge("increment2", END)

app = graph.compile()
result = app.invoke({"count": 0})
print(f"최종 결과 {result}")
app.get_graph().print_ascii()

최종 결과 {'count': 11}
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| increment1 | 
+------------+ 
       *       
       *       
       *       
+------------+ 
| increment2 | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


### 조건부 엣지
 - 런타임 상태에 따라 동적으로 실행 경로 결정
 - add_conditional_edges()

In [ ]:


# 라우팅 키와 노드명은 상수로 관리한다.
# LangGraph 조건부 엣지는 라우터가 반환한 키를 path_map으로 노드에 매핑한다.
ParityRoute = Literal["even", "odd"]

EVEN_ROUTE: ParityRoute = "even"
ODD_ROUTE: ParityRoute = "odd"
EVEN_NODE = "even_handler"
ODD_NODE = "odd_handler"


# State : 입력 숫자와 노드가 생성한 결과
class NumberState(TypedDict):
    number: Required[int]
    result: NotRequired[str]


def get_number(state: NumberState) -> int:
    number = state["number"]
    if type(number) is not int:
        raise TypeError("number는 int 타입이어야 합니다.")
    return number


def classify_parity(number: int) -> ParityRoute:
    return EVEN_ROUTE if number % 2 == 0 else ODD_ROUTE


# 라우터 : 현재 state를 보고 다음 실행 경로를 결정
def route_by_parity(state: NumberState) -> ParityRoute:
    return classify_parity(get_number(state))


# 노드 : result 값만 업데이트
def even_node(state: NumberState) -> dict[str, str]:
    number = get_number(state)
    return {"result": f"{number} 는 짝수 입니다."}


def odd_node(state: NumberState) -> dict[str, str]:
    number = get_number(state)
    return {"result": f"{number} 는 홀수 입니다."}


def build_number_graph():
    graph = StateGraph(NumberState)

    graph.add_node(EVEN_NODE, even_node)
    graph.add_node(ODD_NODE, odd_node)

    graph.add_conditional_edges(
        START,
        route_by_parity,
        {
            EVEN_ROUTE: EVEN_NODE,
            ODD_ROUTE: ODD_NODE,
        },
    )
    graph.add_edge(EVEN_NODE, END)
    graph.add_edge(ODD_NODE, END)

    return graph.compile()


app = build_number_graph()

for number in [14, 15]:
    result = app.invoke({"number": number})
    print(result["result"])

app.get_graph().print_ascii()

14 는 짝수 입니다.
15 는 홀수 입니다.
               +-----------+                
               | __start__ |                
               +-----------+                
              ..           ..               
            ..               ..             
          ..                   ..           
+--------------+           +-------------+  
| even_handler |           | odd_handler |  
+--------------+           +-------------+  
              **           **               
                **       **                 
                  **   **                   
                +---------+                 
                | __end__ |                 
                +---------+                 


In [16]:
from typing import Literal, NotRequired, Required, TypedDict

from langgraph.graph import END, START, StateGraph


GRADE_A_NODE = "grade_a"
GRADE_B_NODE = "grade_b"
GRADE_C_NODE = "grade_c"

GradeNode = Literal["grade_a", "grade_b", "grade_c"]


class ScoreState(TypedDict):
    score: Required[int]
    grade: NotRequired[str]
    message: NotRequired[str]


def grade_a(state: ScoreState) -> dict[str, str]:
    return {"grade": "A", "message": "A 학점"}


def grade_b(state: ScoreState) -> dict[str, str]:
    return {"grade": "B", "message": "B 학점"}


def grade_c(state: ScoreState) -> dict[str, str]:
    return {"grade": "C", "message": "C 학점"}


def route_grade(state: ScoreState) -> GradeNode:
    score = state["score"]

    if score >= 90:
        return GRADE_A_NODE
    if score >= 80:
        return GRADE_B_NODE
    return GRADE_C_NODE


graph = StateGraph(ScoreState)
graph.add_node(GRADE_A_NODE, grade_a)
graph.add_node(GRADE_B_NODE, grade_b)
graph.add_node(GRADE_C_NODE, grade_c)

graph.add_conditional_edges(START, route_grade)

graph.add_edge(GRADE_A_NODE, END)
graph.add_edge(GRADE_B_NODE, END)
graph.add_edge(GRADE_C_NODE, END)

app = graph.compile()

for score in [95, 85, 75]:
    result = app.invoke({"score": score})
    print(f"{score}점: {result['message']}")

app.get_graph().print_ascii()


95점: A 학점
85점: B 학점
75점: C 학점
                     +-----------+                       
                     | __start__ |                       
                   ..+-----------+...                    
               ....         .        ....                
           ....             .            ....            
         ..                 .                ..          
+---------+           +---------+           +---------+  
| grade_a |           | grade_b |           | grade_c |  
+---------+****       +---------+        ***+---------+  
               ****         *        ****                
                   ****     *    ****                    
                       **   *  **                        
                      +---------+                        
                      | __end__ |                        
                      +---------+                        


In [17]:
from typing import Annotated, Literal

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

from langgraph.graph import END, START, StateGraph

parser = StrOutputParser()

translate_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

summarize_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 3문장으로 번역하세요: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

sentiment_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 답하세요.: \n{text}",
            ),
        ]
    )
    | watson_llm
    | parser
)

report_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "아래 분석 결과를 바탕으로 한 줄 최종 보고서를 작성하세요: \n"
                "원문 : {text} \n 번역 : {translated}\n 요약:{summary}\n 감정: {sentiment}",
            ),
        ]
    )
    | watson_llm
    | parser
)

# 상태관리 : text, route, result, reason
SentimentRoute = Literal["positive", "negative", "neutral"]

ANALYZE_NODE = "analyze_sentiment"
POSITIVE_NODE = "positive"
NEGATIVE_NODE = "negative"
NEUTRAL_NODE = "neutral"


class SentimentAnalysis(BaseModel):
    route: SentimentRoute = Field(..., description="감정 분류 결과")
    reason: str = Field(..., description="판단 근거 한 문장")


class SentimentState(BaseModel):
    text: Annotated[str, Field(..., description="분석할 문장")]
    route: SentimentRoute | None = None
    result: str = ""
    reason: str = ""


parser = PydanticOutputParser(pydantic_object=SentimentAnalysis)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 한국어 감정 분석기입니다.\n"
            "텍스트의 전체 의미를 보고 positive, negative, neutral 중 하나로만 분류하세요.\n\n"
            "{format_instructions}",
        ),
        ("human", "{text}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

sentiment_chain = prompt | ChatOllama(model="exaone3.5:2.4b", temperature=0) | parser


def analyze_sentiment(state: SentimentState) -> dict[str, str]:
    analysis = sentiment_chain.invoke({"text": state.text})
    return {"route": analysis.route, "reason": analysis.reason}


def positive_node(state: SentimentState) -> dict[str, str]:
    return {"result": "긍정"}


def negative_node(state: SentimentState) -> dict[str, str]:
    return {"result": "부정"}


def neutral_node(state: SentimentState) -> dict[str, str]:
    return {"result": "중립"}


def route_sentiment_nodes(state: SentimentState) -> SentimentRoute:
    if state.route is None:
        raise ValueError("감정 분석 결과가 없습니다.")
    return state.route


graph = StateGraph(SentimentState)
graph.add_node(ANALYZE_NODE, analyze_sentiment)
graph.add_node(POSITIVE_NODE, positive_node)
graph.add_node(NEGATIVE_NODE, negative_node)
graph.add_node(NEUTRAL_NODE, neutral_node)

graph.add_edge(START, ANALYZE_NODE)
graph.add_conditional_edges(
    ANALYZE_NODE,
    route_sentiment_nodes,
    {
        "positive": POSITIVE_NODE,
        "negative": NEGATIVE_NODE,
        "neutral": NEUTRAL_NODE,
    },
)
graph.add_edge(POSITIVE_NODE, END)
graph.add_edge(NEGATIVE_NODE, END)
graph.add_edge(NEUTRAL_NODE, END)

app = graph.compile()

for text in ["오늘 주가가 너무 떨어졌는데?", "참 잘하는 짓이다.", "글쎄다."]:
    result = app.invoke({"text": text})
    print(f"{text}: {result['result']} - {result['reason']}")

app.get_graph().print_ascii()


ConnectError: [WinError 10061] 대상 컴퓨터에서 연결을 거부했으므로 연결하지 못했습니다

In [18]:
from typing import Annotated, Literal

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

from langgraph.graph import END, START, StateGraph

parser = StrOutputParser()

translate_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

summarize_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 간략하게 요약하세요: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

sentiment_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 답하세요.: \n{text}",
            ),
        ]
    )
    | watson_llm
    | parser
)

report_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "아래 분석 결과를 바탕으로 한 줄 최종 보고서를 작성하세요: \n"
                "원문 : {text} \n 번역 : {translated}\n 요약:{summary}\n 감정: {sentiment}",
            ),
        ]
    )
    | watson_llm
    | parser
)


class AnalysisState(BaseModel):
    text: str
    translated: str = ""
    summary: str = ""
    sentiment: str = ""
    done: bool


def translated_node(state: AnalysisState):
    return {"translated": translate_chain.invoke({"text": state.text})}


def summary_node(state: AnalysisState):
    return {"summary": summarize_chain.invoke({"text": state.text})}


def sentiment_node(state: AnalysisState):
    return {"sentiment": sentiment_chain.invoke({"text": state.text})}


# 그래프 생성
graph = StateGraph(AnalysisState)
graph.add_node("translated", translated_node)
graph.add_node("summarize", summary_node)
graph.add_node("sentiment", sentiment_node)

graph.add_edge(START, "translated")
graph.add_edge("translated", "summarize")
graph.add_edge("summarize", "sentiment")
graph.add_edge("sentiment", END)

# 그래프 실행
app = graph.compile()
result = app.invoke({"text": "Python is great!", "done": False})
print("번역 : ", result["translated"])
print("요약 : ", result["summary"])
print("감정 : ", result["sentiment"])

app.get_graph().print_ascii()

번역 :  파이썬은 정말 멋져요!
요약 :  Python은 놀라운 프로그래밍 언어입니다!
감정 :  긍정
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| translated | 
+------------+ 
       *       
       *       
       *       
+-----------+  
| summarize |  
+-----------+  
       *       
       *       
       *       
+-----------+  
| sentiment |  
+-----------+  
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


In [20]:
class SummaryState(BaseModel):
    text: str
    summary: str = ""
    quality: str = ""
    retries: int = 0


def summary_node(state: SummaryState):
    return {"summary": summarize_chain.invoke({"text": state.text})}


def check_quality_node(state: SummaryState):
    quality = "good" if len(state.summary) >= 100 else "poor"
    return {"quality": quality}


def retry_node(state: SummaryState):
    return {"retries": state.retries + 1}


def route_by_quality(state: SummaryState):
    if state.quality == "poor" and state.retries < 3:
        return "retry"
    return "done"


graph = StateGraph(SummaryState)

graph.add_node("summary", summary_node)
graph.add_node("check_quality", check_quality_node)
graph.add_node("retry", retry_node)

graph.add_edge(START, "summary")
graph.add_edge("summary", "check_quality")

graph.add_conditional_edges(
    "check_quality", route_by_quality, {"retry": "retry", "done": END}
)

graph.add_edge("retry", "summary")

app = graph.compile()

result = app.invoke({"text": "짧은 글"})

print(f"최종 요약({len(result['summary'])}자): {result['summary']}")
app.get_graph().print_ascii()

최종 요약(93자): 짧은 글을 요약하려면, 핵심 아이디어와 중요한 세부 사항을 파악하고, 불필요한 정보를 제거해야 합니다. 요약은 원본 글의 의미를 유지하면서도 간결하게 전달해야 합니다.
              +-----------+          
              | __start__ |          
              +-----------+          
                    *                
                    *                
                    *                
              +---------+            
              | summary |            
              +---------+            
             ***         ***         
            *               *        
          **                 ***     
+---------------+               *    
| check_quality |               *    
+---------------+.              *    
        .         .....         *    
        .              ...      *    
        .                 ...   *    
   +---------+             +-------+ 
   | __end__ |             | retry | 
   +---------+             +-------+ 


In [ ]:
class VerifyState(BaseModel):
    question: str = ""
    answer: str
    feedback: str
    is_verified: bool
    attempt: int


def generate(state: VerifyState):
    """답변을 생성합니다. 이전 피드백이 있다면 반영합니다."""
    prompt = f"질문 : {state.question}"

    if state.feedback:
        prompt += f"\n이전 답변의 피드백 : {state.feedback} \n 위 피드백을 반영하여 개선된 답변을 작성하세요"

    result = watson_llm.invoke(prompt)

    return {"answer": result.content, "attempt": state.attempt + 1}


def verify(state: VerifyState):
    """답변의 정확성과 완전성을 검증합니다."""
    verification = watson_llm.invoke(f"""
    다음 답변의 정확성과 완전성을 검증하세요
                      
    질문:
    {state.question}

    답변:
    {state.answer}

    정확하고 완전하면 첫 줄에 'PASS'를, 수정이 필요하면 첫 줄에 'FAIL'을 쓰고 구체적인 개선 사항을 설명하세요.
    """)

    content = verification.content
    is_pass = content.strip().startswith("PASS")

    return {"is_verified": is_pass, "feedback": content}


def should_retry(state: VerifyState):
    """검증 통과 또는 최대 횟수 도달시 종료"""
    if state.is_verified or state.attempt >= 3:
        return END
    return "generate"


graph = StateGraph(VerifyState)
graph.add_node("generate", generate)
graph.add_node("verify", verify)

graph.add_edge(START, "generate")
graph.add_edge("generate", "verify")
graph.add_conditional_edges("verify", should_retry, {"generate": "generate", END: END})

app = graph.compile()
result = app.invoke(
    {
        "question": "파이썬에서 GIL이 무엇이며 멀티쓰레딩에 어떤 영향을 주는지 설명해라",
        "answer": "",
        "feedback": "",
        "is_verified": False,
        "attempt": 0,
    }
)

print("시도횟수", result["attempt"])
print("검증통과 여부", result["is_verified"])
print("최종답변", result["answer"])

시도횟수 1
검증통과 여부 True
최종답변 질문에 대한 답변은 다음과 같습니다:

파이썬에서 GIL(Global Interpreter Lock)은 파이썬 인터프리터가 동시에 하나의 스레드만 실행하도록 제한하는 락입니다. 이는 파이썬 인터프리터가 메모리 관리를 담당하는 CPython 구현에서 발생하는 문제를 해결하기 위해 도입되었습니다. GIL은 파이썬의 메모리 관리가 스레드 안전하지 않기 때문에 동시에 여러 스레드가 파이썬 객체를 수정하는 것을 방지합니다.

멀티쓰레딩에 미치는 영향은 다음과 같습니다:

1. **CPU 바운드 작업에 대한 성능 저하**: GIL은 멀티코어 프로세서에서 멀티쓰레딩의 이점을 제한합니다. 파이썬 프로그램이 CPU 바운드 작업(즉, 계산이 많이 필요한 작업)을 수행하는 경우, GIL은 여러 스레드가 동시에 실행되는 것을 방지하여 멀티코어 프로세서의 이점을 활용할 수 없게 합니다. 이는 파이썬 프로그램이 멀티코어 시스템에서 기대한 만큼의 성능 향상을 얻지 못하게 하는 원인이 됩니다.

2. **I/O 바운드 작업에 대한 이점**: 반면에, GIL은 I/O 바운드 작업(즉, 입출력 작업이 많이 필요한 작업)에 대해서는 큰 문제가 되지 않습니다. 파이썬 스레드가 I/O 작업을 기다리는 동안 GIL을 해제하여 다른 스레드가 실행될 수 있게 합니다. 이는 웹 서버와 같은 I/O 작업이 많은 애플리케이션에서 멀티쓰레딩이 여전히 유용할 수 있음을 의미합니다.

3. **멀티프로세싱 대안**: GIL로 인한 멀티쓰레딩의 제한을 극복하기 위해, 파이썬에서는 멀티프로세싱을 사용할 수 있습니다. 멀티프로세싱은 각 프로세스가 자체 파이썬 인터프리터와 메모리 공간을 가지므로, GIL의 영향을 받지 않습니다. 이는 CPU 바운드 작업을 병렬로 실행하고자 할 때 특히 유용합니다.

4. **GIL의 미래**: 파이썬 커뮤니티는 GIL에 대한 논의를 지속적으로 진행하고 있으며, GIL을 제거하거나 대체할 수 있는 방법을 탐구하고 있습니다. 그러나 GIL을 제거

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import time

parser = StrOutputParser()

translate_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

summarize_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 간략하게 요약하세요. 요약문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

keyword_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "다음 텍스트에서 주요 키워드 5개를 추출하세요. 키워드만 출력: \n{text}",
            ),
        ]
    )
    | watson_llm
    | parser
)


def with_log(label, chain):
    def run(inputs):
        print(f"{label} 시작")
        time.sleep(3)
        result = chain.invoke(inputs)
        print(f"{label} 종료")
        return result

    return RunnableLambda(run)


parallel_chain = RunnableParallel(
    translated=with_log("번역", translate_chain),
    summary=with_log("요약", summarize_chain),
    keywords=with_log("키워드 추출", keyword_chain),
)

text = {"text": "Python is a versatile language used in AI and web development"}

start_time = time.perf_counter()
result = parallel_chain.invoke(text)
elapsed_time = time.perf_counter() - start_time

print(f"실행 시간: {elapsed_time:.2f}초")
print("번역", result["translated"][:100])
print("요약", result["summary"][:100])
print("키워드", result["keywords"][:100])


번역 시작
요약 시작
키워드 추출 시작
키워드 추출 종료
요약 종료
번역 종료
실행 시간: 3.57초
번역 파이썬은 AI와 웹 개발에서 사용되는 다용도 언어입니다.
요약 파이썬은 AI와 웹 개발에 사용되는 다용도 언어입니다.
키워드 Python, versatile, language, AI, web development


In [ ]:
import sys

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_ibm import WatsonxEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("./data/Summary of ChatGPTGPT-4 Research.pdf")
# STEP 2 : 문서분할
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(loader.load())
print(f"chunks 수 {chunks}")

# STEP 3 : 인덱싱 - 임베딩
embeddings = watson_embedding
# STEP 4: 벡터스토어(Chroma or FAISS)
vectorstore = Chroma.from_documents(
    chunks, embeddings, persist_directory="./db/chroma_db", collection_name="research"
)
# STEP 5 : as_retriever() : Vector store를 Retriever 형태로 변환하여 LangChain에 연결
retrivever = vectorstore.as_retriever(search_type="similarity", serach_kwargs={"k": 3})


chunks 수 [Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-05T00:33:07+00:00', 'author': '', 'keywords': '', 'moddate': '2023-04-05T00:33:07+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/Summary of ChatGPTGPT-4 Research.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Summary of ChatGPT/GPT-4 Research\nand Perspective Towards the Future of Large\nLanguage Models\nYiheng Liu ∗1, Tianle Han ∗1, Siyuan Ma 1, Jiayue Zhang 1,\nYuanyuan Yang1, Jiaming Tian 1, Hao He 1, Antong Li 2, Mengshen\nHe1, Zhengliang Liu 3, Zihao Wu 3, Dajiang Zhu 4, Xiang Li 5, Ning\nQiang1, Dingang Shen 6,7,8, Tianming Liu 3, and Bao Ge †1\n1School of Physics and Information Technology, Shaanxi Normal University, Xi’an\n710119 China'), Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX w

In [ ]:
from typing import List
from pydantic import BaseModel
from langchain_community.vectorstores import Chroma

from langchain_core.documents import Document


class RAGState(BaseModel):
    query: str
    retrieved_docs: List[Document] = None
    answer: str = ""


def retrieve(state: RAGState):
    # 기존의 벡터스토어에 질의
    vectorstore = Chroma(
        collection_name="research",
        embedding_function=watson_embedding,
        persist_directory="./db/chroma_db/",
    )

    docs = vectorstore.similarity_search(state.query, k=3)

    return {"retrieved_docs" : docs}


def generate(state: RAGState):

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = """\
            다음 컨텍스트를 참고하여 질문에 답하세요.
            컨텍스트에 없는 내용은 모른다고 답하세요.

            컨텍스트:
            {context}

            질문:
            {query}
        """
    
    response = watson_llm.invoke(prompt.format(context=context, query=state.query))

    return {"answer" : response.content}

graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
result = app.invoke({"query" : "where can i use chatGPT?"})
print(result['answer'])
app.get_graph().draw_ascii()


C:\Users\soldesk\AppData\Local\Temp\ipykernel_4448\3893268473.py:16: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


ChatGPT는 인터넷에 연결된 장치에서 사용할 수 있습니다. 웹 브라우저를 통해 OpenAI의 공식 웹사이트에서 직접 접근할 수 있으며, 또한 다양한 애플리케이션과 서비스에서도 통합되어 사용될 수 있습니다. 하지만, 제공된 컨텍스트에는 ChatGPT의 사용 가능한 구체적인 장소나 방법에 대한 정보가 없으므로, 추가적인 정보가 필요합니다.


'+-----------+  \r\n| __start__ |  \r\n+-----------+  \r\n      *        \r\n      *        \r\n      *        \r\n+----------+   \r\n| retrieve |   \r\n+----------+   \r\n      *        \r\n      *        \r\n      *        \r\n+----------+   \r\n| generate |   \r\n+----------+   \r\n      *        \r\n      *        \r\n      *        \r\n +---------+   \r\n | __end__ |   \r\n +---------+   '

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document

class RAGState(BaseModel):
    query: str = ""
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""
    retry_count: int = 0
    is_relevant: bool = False

def evaluate(state: RAGState):
    """답변이 질문과 관련이 있는지 평가"""
    prompt = ChatPromptTemplate.from_template(
    """
    질문:
    {query}

    답변
    {answer}

    이 답변이 질문에 적절히 대답하고 있나요? 'yes' 또는 'no'로 답하세요.
    """
    )

    response = watson_llm.invoke(prompt.format(query=state.query, answer=state.answer))
    is_relevant = response.content.strip().lower().startswith("yes")

    return {"is_relevant": is_relevant, "retry_count": state.retry_count + 1}

def should_retry(state:RAGState):
    """재검색 여부 결정"""
    if state.is_relevant or state.retry_count >=2:
        return "done"
    return "retry"

# 그래프 구성

graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", should_retry, {"done":END, "retry":"retrieve"})

app = graph.compile()
result = app.invoke({"query": "where can i use chatGPT?", "retry_count": 0})
print(result["answer"])
app.get_graph().draw_ascii()

C:\Users\soldesk\AppData\Local\Temp\ipykernel_22344\2838437069.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


ChatGPT는 인터넷에 연결된 장치에서 사용할 수 있습니다. 웹 브라우저를 통해 OpenAI의 공식 웹사이트에서 직접 접근할 수 있으며, 또한 다양한 애플리케이션과 서비스에서도 통합되어 사용될 수 있습니다. 하지만, 제가 가지고 있는 정보는 2021년 9월까지이므로, 그 이후에 발표된 새로운 플랫폼이나 서비스에 대해서는 알지 못할 수 있습니다.


'           +-----------+       \r\n           | __start__ |       \r\n           +-----------+       \r\n                  *            \r\n                  *            \r\n                  *            \r\n            +----------+       \r\n            | retrieve |       \r\n            +----------+       \r\n           ***        ...      \r\n          *              .     \r\n        **                ...  \r\n+----------+                 . \r\n| generate |              ...  \r\n+----------+             .     \r\n           ***        ...      \r\n              *      .         \r\n               **  ..          \r\n            +----------+       \r\n            | evaluate |       \r\n            +----------+       \r\n                  .            \r\n                  .            \r\n                  .            \r\n            +---------+        \r\n            | __end__ |        \r\n            +---------+        '

### 검색 품질 개선
- multi-query: 여러 관점의 쿼리로 검색 (모호한 질문, 넓은 검색 범위 필요)
- HyDE : 가상 문서를 생성하여 검색 (쿼리와 문서의 표현 차이가 큰 경우)
- Self-RAG : 답변을 평가하고 반복 개선

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

class MultiQueryState(BaseModel):
    query: str = ""
    sub_queries: List[str] = Field(default_factory=list)
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""

def generate_sub_queries(state: MultiQueryState):
    """원본 질문을 여러 관점 하위 쿼리로 분해"""
    prompt = """\
    다음 질문에 대해 서로 다른 관점의 검색 쿼리 3개를 생성하세요.
    각 쿼리를 줄바꿈으로 구분하세요.

    원본 질문:
    {query}
    """

    response = watson_llm.invoke(prompt.format(query=state.query))

    # sub query
    sub_queries = [ q for q in response.content.split("\n") if q.strip()]
    return {"sub_queries": sub_queries}

def multi_retrieve(state: MultiQueryState):
    """각 하위 쿼리로 검색하고 결과 합치기"""
    vectorstore = Chroma(
        collection_name="research",
        embedding_function=watson_embedding,
        persist_directory="./db/chroma_db/",
    )

    all_docs = []
    seen_contents = set()
    for sub_query in state.sub_queries:
        docs = vectorstore.similarity_search(sub_query, k=3)
        for doc in docs:
            if doc.page_content not in seen_contents:
                all_docs.append(doc)
                seen_contents.add(doc.page_content)

    return {"retrieved_docs": all_docs}


def generate(state: MultiQueryState):

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = """\
            다음 컨텍스트를 참고하여 질문에 답하세요.
            컨텍스트에 없는 내용은 모른다고 답하세요.

            컨텍스트:
            {context}

            질문:
            {query}
        """

    response = watson_llm.invoke(prompt.format(context=context, query=state.query))

    return {"answer": response.content}


graph = StateGraph(MultiQueryState)

graph.add_node("generate_sub_queries", generate_sub_queries)
graph.add_node("retrieve", multi_retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "generate_sub_queries")
graph.add_edge("generate_sub_queries", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
result = app.invoke({"query": "where can i use chatGPT?"})
print(result["answer"])

print(app.get_graph().draw_ascii())

ChatGPT는 인터넷에 연결된 장치에서 사용할 수 있습니다. 웹 브라우저를 통해 OpenAI의 공식 웹사이트에서 직접 접근할 수 있으며, 또한 다양한 애플리케이션과 서비스에서도 통합되어 사용될 수 있습니다. 하지만, 제가 가지고 있는 정보는 2021년 9월까지이므로, 그 이후에 발표된 새로운 플랫폼이나 서비스에 대해서는 알지 못할 수 있습니다.
      +-----------+      
      | __start__ |      
      +-----------+      
            *            
            *            
            *            
+----------------------+ 
| generate_sub_queries | 
+----------------------+ 
            *            
            *            
            *            
      +----------+       
      | retrieve |       
      +----------+       
            *            
            *            
            *            
      +----------+       
      | generate |       
      +----------+       
            *            
            *            
            *            
      +---------+        
      | __end__ |        
      +---------+        


# HyDE
# 질문 => 가상의 정답 문서를 먼저 생성, 가상의 문서를 검색어처럼 사용하여 더 관련성 높은 문서 찾음

In [28]:
from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

class HyDEState(BaseModel):
    query: str = ""
    hypothetical_doc: str = ""
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""


def hyde_retrieve(state: HyDEState):
    # 가상 문서를 쿼리로 사용하여 검색
    vectorstore = Chroma(
        collection_name="research",
        embedding_function=watson_embedding,
        persist_directory="./db/chroma_db/",
    )

    search_query = state.hypothetical_doc.strip()[:1000]
    docs = vectorstore.similarity_search(search_query, k=3)

    return {"retrieved_docs": docs}


def generate_hypothetical(state:HyDEState):
    """질문에 대한 가상의 답변 문서를 생성합니다."""

    prompt = """\
        다음 질문에 대한 검색용 가상 문서를 작성하세요.
        3문장 이내로 짧게 작성하세요.
        핵심 키워드와 관련 분야만 포함하세요.

        질문:
        {query}
    """

    response = watson_llm.invoke(prompt.format(query=state.query))
    hypothetical_doc = response.content.strip()[:1000]
    return {"hypothetical_doc": hypothetical_doc}


def generate(state: HyDEState):

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = """\
        다음 컨텍스트를 참고하여 질문에 답하세요.
        컨텍스트에 없는 내용은 모른다고 답하세요.

        컨텍스트:
        {context}

        질문:
        {query}
    """

    response = watson_llm.invoke(prompt.format(context=context, query=state.query))
    return {"answer": response.content}

graph = StateGraph(HyDEState)
graph.add_node("hypothetical", generate_hypothetical)
graph.add_node("retrieve", hyde_retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "hypothetical")
graph.add_edge("hypothetical", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
result = app.invoke(
    {"query": "ChatGPT and GPT-4 have potential applications in which domains?"}
)
print(result["answer"])

print(app.get_graph().draw_ascii())

제공된 컨텍스트에 따르면, ChatGPT와 GPT-4는 다양한 분야에서 잠재적인 응용 가능성을 가지고 있습니다. 이 논문은 이러한 모델들이 다양한 분야에서 어떻게 활용될 수 있는지에 대한 포괄적인 조사를 제공하고자 합니다. 구체적인 분야에 대한 언급은 컨텍스트에 포함되어 있지 않으므로, 모든 가능한 분야를 열거하기는 어렵습니다. 하지만, 이 모델들이 다양한 분야에서 활용될 수 있다는 것을 알 수 있습니다.
  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
+--------------+ 
| hypothetical | 
+--------------+ 
        *        
        *        
        *        
  +----------+   
  | retrieve |   
  +----------+   
        *        
        *        
        *        
  +----------+   
  | generate |   
  +----------+   
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [32]:
## self refine tag
# 답변을 생성 => 평가 => 재검색 or 답변 수정

from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document


class SelfRefineRAGState(BaseModel):
    query: str = ""
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""
    evaluation: str = ""
    retry_count: int = 0
    is_relevant: bool = False


def retrieve(state: RAGState):
    vectorstore = Chroma(
    collection_name="research",
    embedding_function=watson_embedding,
    persist_directory="./db/chroma_db/",
    )
    docs = vectorstore.similarity_search(state.query, k=3)

    return {"retrieved_docs": docs}


def generate(state: RAGState):

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = """\
            다음 컨텍스트를 참고하여 질문에 답하세요.
            컨텍스트에 없는 내용은 모른다고 답하세요.
            컨텍스트에 정보가 부족하면 그 사실을 명시하에쇼.

            컨텍스트:
            {context}

            질문:
            {query}
        """

    response = watson_llm.invoke(prompt.format(context=context, query=state.query))

    return {"answer": response.content}


def evaluate(state: SelfRefineRAGState):
    """생성한 답변이 답변의 충실도와 관련성을 평가"""

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = ChatPromptTemplate.from_template(
        """
    질문:
    {query}

    컨텍스트:
    {context}

    답변:
    {answer}

    반드시 아래 둘 중 하나로만 출력하세요
    'sufficient', 'insufficient'
    """
    )

    response = watson_llm.invoke(
        prompt.format(query=state.query, answer=state.answer, context=context)
    )

    content = response.content.lower().strip()
    evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

    return {"evaluation": evaluation}


def refine(state: SelfRefineRAGState):
    """평가 결과를 반영하여 답변을 개선한다."""
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = """\
        다음 답변을 개선하세요.
        원래 질문 : {query}
        컨텍스트 : {context}
        이전 답변 : {answer}
        컨텍스트에 더 충실하고 질문에 더 정확히 답하도록 수정하세요.
    """

    response = watson_llm.invoke(
        prompt.format(context=context, query=state.query, answer=state.answer)
    )
    return {"answer": response.content, "retry_count": state.retry_count + 1}


def route_after_eval(state: SelfRefineRAGState):
    """재검색 여부 결정"""
    if state.evaluation == "sufficient" or state.retry_count >= 2:
        return "done"
    return "refine"


# 그래프 구성
graph = StateGraph(SelfRefineRAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("refine", refine)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate", route_after_eval, {"done": END, "refine": "refine"}
)
graph.add_edge("refine", "evaluate")

app = graph.compile()
result = app.invoke({"query": "where can i use chatGPT?", "retry_count": 0})
print(result["answer"])
print(app.get_graph().draw_ascii())

개선된 답변: 제공된 컨텍스트에 따르면, ChatGPT는 다음과 같은 분야에서 사용될 수 있습니다:

1. 개발 분야: ChatGPT는 코드 생성 및 개발 관련 질문에 대한 답변을 제공하는 데 사용될 수 있습니다. 그러나 현재 훈련 데이터가 Python, C++, Java와 같은 프로그래밍 언어에 편중되어 있어 다른 프로그래밍 언어나 코딩 스타일에는 적합하지 않을 수 있습니다. 또한, ChatGPT는 소규모 범위(48)에서 약간 약한 성능을 보이기도 합니다.

2. 교육 분야: 학생들은 ChatGPT를 사용하여 다양한 학문 분야(예: 물리학, 수학, 화학 등)에서 질문에 대한 답변을 학습, 비교 및 검증할 수 있습니다. 예를 들어, Chen et al. [25]는 ChatGPT를 대화 에이전트로 사용하여 장면, 시간, 캐릭터 속성 및 캐릭터 관계를 포함하는 대화 데이터셋(HPD)을 구축했습니다. 그러나 ChatGPT의 테스트 세트에서의 성능은 미흡하며 개선의 여지가 있습니다. 또한, ChatGPT는 복잡한 텍스트를 단순화하는 능력을 보여주었으며, 이를 위해 세 가지 픽션 예제를 제공했습니다.

컨텍스트에는 ChatGPT가 사용될 수 있는 다른 분야에 대한 정보가 없으므로, 추가적인 정보가 필요한 경우 해당 분야에 대한 정보를 찾아보시기 바랍니다.
         +-----------+           
         | __start__ |           
         +-----------+           
                *                
                *                
                *                
          +----------+           
          | retrieve |           
          +----------+           
                *                
         

In [33]:
## self rag
# 답변을 생성 => 평가 => 부족 => 질문 개선 => 검색 => 새 문서 답변

from typing import List
from pydantic import BaseModel, Field
from langchain_core.documents import Document


class SelfRAGState(BaseModel):
    query: str = ""
    retrieved_docs: List[Document] = Field(default_factory=list)
    answer: str = ""
    evaluation: str = ""
    retry_count: int = 0


def retrieve(state: SelfRAGState):
    vectorstore = Chroma(
        collection_name="research",
        embedding_function=watson_embedding,
        persist_directory="./db/chroma_db/",
    )
    docs = vectorstore.similarity_search(state.query, k=3)

    return {"retrieved_docs": docs}


def generate(state: SelfRAGState):

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = """\
            다음 컨텍스트를 참고하여 질문에 답하세요.
            컨텍스트에 없는 내용은 모른다고 답하세요.
            컨텍스트에 정보가 부족하면 그 사실을 명시하세요.

            컨텍스트:
            {context}

            질문:
            {query}
        """

    response = watson_llm.invoke(prompt.format(context=context, query=state.query))

    return {"answer": response.content}


def evaluate(state: SelfRefineRAGState):
    """생성한 답변이 답변의 충실도와 관련성을 평가"""

    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)

    prompt = ChatPromptTemplate.from_template(
        """
    질문:
    {query}

    컨텍스트:
    {context}

    답변:
    {answer}

    반드시 아래 둘 중 하나로만 출력하세요
    'sufficient', 'insufficient'
    """
    )

    response = watson_llm.invoke(
        prompt.format(query=state.query, answer=state.answer, context=context)
    )

    content = response.content.lower().strip()
    evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

    return {"evaluation": evaluation}


def rewrite_query(state: SelfRefineRAGState):
    """평가 결과를 반영하여 답변을 개선한다."""
    prompt = """\
        원래 질문 : {query}
        검색 결과가 충분하지 않았습니다.
        더 구체적이고 검색하기 좋은 질문으로 재작성하세요.
        질문만 출력하세요.
    """

    response = watson_llm.invoke(
        prompt.format(query=state.query)
    )
    return {"answer": response.content, "retry_count": state.retry_count + 1}


def route_after_eval(state: SelfRefineRAGState):
    """재검색 여부 결정"""
    if state.evaluation == "sufficient" or state.retry_count >= 2:
        return "done"
    return "retry"


# 그래프 구성
graph = StateGraph(SelfRAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("rewrite", rewrite_query)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges(
    "evaluate", route_after_eval, {"done": END, "retry": "rewrite"}
)
graph.add_edge("rewrite", "retrieve")

app = graph.compile()
result = app.invoke({"query": "where can i use chatGPT?", "retry_count": 0})
print(result["answer"])
print(app.get_graph().draw_ascii())


Based on the provided context, ChatGPT can be used in the following areas:

1. Education field: ChatGPT is commonly used for question and answer testing in the education sector. Users can use ChatGPT to learn, compare and verify answers for different academic subjects such as physics, mathematics, and chemistry.

2. Code generation: ChatGPT can be applied in the field of code generation. However, there are still several challenges, such as its limited application scope due to training data being biased towards programming languages like Python, C++, and Java, making it potentially unsuitable for some programming languages or coding styles.

3. Dialogue generation: ChatGPT can be used as a conversational agent to generate dialogue. In the study mentioned, a dialogue dataset (HPD) with scenes, timelines, character attributes, and character relationships was constructed to use ChatGPT for this purpose. However, its performance on the test set was poor, and there is room for improvement.



### LLM 애플리케이션 성능 최적화 전략


In [39]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache
import time


text = "파이썬이란 무엇인가?"

print("====캐시 없음====")
set_llm_cache(InMemoryCache())

start = time.time()
r1 = watson_llm.invoke(text)
print(f"1회차 : {time.time()-start:.2f}초")

start = time.time()
r2 = watson_llm.invoke(text)
print(f"2회차 : {time.time() - start:.2f}초")

# InMemoryCache
print("=========SQLiteCache=============")
set_llm_cache(SQLiteCache(database_path="./db/llm_cache.db"))

start = time.time()
r3 = watson_llm.invoke("LangChain 이란?")
print(f"3회차 : {time.time() - start:.2f}초")

start = time.time()
r4 = watson_llm.invoke("LangChain 이란?")
print(f"4회차 : {time.time() - start:.2f}초")


====캐시 없음====
1회차 : 5.30초
2회차 : 0.00초
=========SQLiteCache=============
3회차 : 2.19초
4회차 : 0.01초


c:\source\ollama\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
